# Data Challenge 12 — Intro to Logistic Regression

**Hook (Attention Grabber)**  
> “If an app told a restaurant it has an 80% chance of getting an **A** on inspection, would you trust it?”

**Learning Goals**
- Show why **linear regression** is a bad fit for a **binary (0/1)** target.
- Fit a **one-feature logistic regression** and interpret probabilities.
- Extend to a **two-feature logistic model with standardized inputs**.
- Communicate results using **AWES** and discuss **ethics & people impact**.

**Data:** June 1, 2025 - Nov 4, 2025 Restaurant Health Inspection

[Restaurant Health Inspection](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (quick links):**
- Train/Test Split — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- LinearRegression — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- LogisticRegression — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- StandardScaler — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- accuracy_score — scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html
- corr — pandas: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html

### Pseudocode Plan (Linear vs Logistic + Scaling)
1) **Load CSV** → preview shape/columns; keep needed fields.  
2) **Engineer binary Y**: `is_A = 1 if grade == 'A' else 0`.  
3) **Pick numeric X**:  
   - **X1:** `score` (inspection score; lower is better)  
   - **X2:** `critical_num = 1 if critical_flag == 'Critical' else 0` (for extension)  
4) **Split** → `X_train, X_test, y_train, y_test` (70/30, stratify by Y, fixed random_state).  
5) **Model A (Incorrect)** → **LinearRegression** on Y~X1:  
   - Report **MSE**, **R²**, count predictions **<0 or >1**,  
6) **Model B (Correct)** → **LogisticRegression** on Y~X1:  
   - Report **Accuracy**
7) **Visual (OPTIONAL)** → scatter Y vs X1 with **linear line** vs **logistic sigmoid** curve  
8) **Extension** → scale X1+X2 with **StandardScaler**; fit **LogisticRegression**:  
   - Compare **Accuracy** to one-feature logistic  
9) **Interpret** → 2–3 sentences on why linear fails and how logistic fixes it  


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

## Step 1 — Imports and Plot Defaults

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression



### Step 2 — Load CSV & Preview
- Point to your New York City Restaurant Inspection Data 

In [32]:
df = pd.read_csv("/Users/Marcy_Student/Downloads/DOHMH_New_York_City_Restaurant_Inspection_Results_20251104 copy.csv")

# Preview
print(df.shape)
df.head()
df.dtypes


(291278, 27)


CAMIS                      int64
DBA                       object
BORO                      object
BUILDING                  object
STREET                    object
ZIPCODE                  float64
PHONE                     object
CUISINE DESCRIPTION       object
INSPECTION DATE           object
ACTION                    object
VIOLATION CODE            object
VIOLATION DESCRIPTION     object
CRITICAL FLAG             object
SCORE                    float64
GRADE                     object
GRADE DATE                object
RECORD DATE               object
INSPECTION TYPE           object
Latitude                 float64
Longitude                float64
Community Board          float64
Council District         float64
Census Tract             float64
BIN                      float64
BBL                      float64
NTA                       object
Location                  object
dtype: object

## Step 3 — Clean and Engineer Features
- Make sure `SCORE` is numeric and do any other data type clean-up 
- Engineer binary target variable (Y) based on instructor guidance above `is_A`
- Engineer binary predictor (X) based on instructor guidance above `critical_num`


In [33]:
df_small = df[['SCORE', 'GRADE', 'CRITICAL FLAG']]

# Ensure SCORE is numeric and drop rows with missing values
df_small['SCORE'] = pd.to_numeric(df_small['SCORE'], errors='coerce')
df_small = df_small.dropna(subset=['SCORE', 'GRADE', 'CRITICAL FLAG'])

# Binary is_A
df_small['is_A'] = (df_small['GRADE'] == 'A').astype(int)

# 1 if inspection is Critical, else 0
df_small['critical_num'] = (df_small['CRITICAL FLAG'].str.strip().str.lower() == 'critical').astype(int)

df_small.head()



/var/folders/7n/rj4gbkk13_1f9n1qf54j12gh0000gp/T/ipykernel_42876/3214569415.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small['SCORE'] = pd.to_numeric(df_small['SCORE'], errors='coerce')


,SCORE,GRADE,CRITICAL FLAG,is_A,critical_num
18,13.0,A,Critical,1,1
36,13.0,A,Not Critical,1,0
37,0.0,P,Not Applicable,0,0
54,0.0,A,Not Applicable,1,0
56,0.0,Z,Not Applicable,0,0


## Step 4 — Split Data (70/30 Stratify by Target)

In [34]:
X = df_small[['SCORE']]
y = df_small['is_A']

# Spliting data 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("Training size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Proportion of A in train:", y_train.mean().round(3))
print("Proportion of A in test:", y_test.mean().round(3))


Training size: 99535
Test size: 42659
Proportion of A in train: 0.677
Proportion of A in test: 0.677


## Step 5 — Model A: Linear Regression on a Binary Target (Incorrect)

- Fit `is_A (Y var) ~ SCORE (X pred)` using **LinearRegression**  
- Report **MSE**, **R²**, and how many predictions fall outside [0, 1]  
- Estimate accuracy by thresholding predictions at 0.5 (done for you but understand the code) 

💡 Hint:  
`accuracy_score(y_test, (y_pred >= 0.5).astype(int))`

In [35]:
lin_reg = LinearRegression().fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
outofbounds = np.sum((y_pred < 0) | (y_pred > 1))
accuracy = accuracy_score(y_test, (y_pred >= 0.5).astype(int))

print(f"MSE: {mse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Predictions outside [0,1]: {outofbounds}")
print(f"Accuracy with thresholded of 0.5: {accuracy:.3f}")


MSE: 0.109
R²: 0.503
Predictions outside [0,1]: 3832
Accuracy with thresholded of 0.5: 0.884


## Step 6 — Model B: Logistic Regression (One Feature)

- Fit `is_A ~ score` using **LogisticRegression**  
- Compute predictions with `.predict()`  
- Evaluate accuracy with `accuracy_score()`

In [36]:
log_reg = LogisticRegression().fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
accuracy_log = accuracy_score(y_test, y_pred_log)

print(f"Logistic Regression Model Accuracy: {accuracy_log:.3f}")


Logistic Regression Model Accuracy: 0.970


## Step 7 (OPTIONAL) — Visual Comparison: Linear vs Logistic


## Step 8 — Logistic Regression with Two **Scaled** Features

- Use `SCORE` and `critical_num` as your two X predictors that need to be scaled
- Look at documentation above to see how you would fit a StandardScalar() object 


In [37]:
# Two features
X2 = df_small[['SCORE', 'critical_num']]
y2 = df_small['is_A']

# Split
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=42)

scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

# Log_reg with 2 features
log_reg2 = LogisticRegression().fit(X2_train_scaled, y2_train)

y2_pred = log_reg2.predict(X2_test_scaled)
acc2 = accuracy_score(y2_test, y2_pred)

print(f"Two-feature Logistic Regression Accuracy: {acc2:.3f}")


Two-feature Logistic Regression Accuracy: 0.970


# We Share — Reflection & Wrap-Up

Write **two short paragraphs** (4–6 sentences each). Be specific and use evidence from your notebook.

1️⃣ **How do you know Linear Regression was a poor model choice for this task?**  
Describe what you observed in your results or plots that showed it didn’t work well for a binary outcome.  
Consider: Were predictions outside 0–1? Did the fit look wrong? What happened when you used 0.5 as a cutoff?  
Connect this to the idea that classification models should output probabilities between 0 and 1.

2️⃣ **When should we scale features in logistic regression (and when not to)?**  
Explain what scaling does, and why it might (or might not) matter for different kinds of features.  
Use this project to reason through whether `score` and `critical_num` needed scaling.  
Hint: Think about what “continuous” vs “binary” means for scaling decisions.

1. 
Linear Regression was a poor model choice for this binary classification because many of its predictions fell outside the valid probability range of 0–1. In my results, 3,832 predictions were below 0 or above 1. Even though the accuracy after applying a 0.5 cutoff was 0.884, the R² value of 0.503 and MSE of 0.109 suggested the model did not fit the data well. This reinforced that Linear Regression isn't ment for binary outcomes, since classification models like Logistic Regression naturally produce probabilities between 0 and 1 and handle decision boundaries more effectively.

2. 
Feature scaling in Logistic Regression is helpful when continuous predictors vary widely in range, because scaling standardizes their influence and helps the algorithm converge efficiently. However, scaling isn’t needed for binary or categorical features, since they already take on small, consistent values (0 or 1). The score variable were continuous so benefited from scaling to ensure its coefficients were on a similar scale to other variables. In contrast, critical_num was binary, so scaling it wouldn't change its interpretation or improve model performance. In short, scaling was useful for continuous features like SCORE, but unnecessary for binary ones like critical_num.